In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [ ]:
training_data = datasets.MNIST(
    "data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.MNIST(
    "data",
    train=False,
    download=True,
    transform=ToTensor()
)
# print(test_data)

In [ ]:
BATCH_SIZE = 128

train_dataloader = DataLoader(
    training_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_dataloader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE
)

for x, y in train_dataloader:
    print(f"shape of x[N, C, W, H]:{x.shape}")
    print(f"share of y: {y.shape}, {y.dtype}")
    print(f"lable of y: {y[0].item()}")
    break

In [ ]:
from torch.cuda import is_available
from torch.nn.modules.linear import Linear
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.nmodel = nn.Sequential(
            nn.Linear(28*28, 256),
            nn.BatchNorm1d(256), # 后加优化用
            nn.ReLU(),
            nn.Dropout(0.5), # 后加优化用
            nn.Linear(256, 128),
            nn.BatchNorm1d(128), # 后加优化用
            nn.ReLU(),
            nn.Dropout(0.2), # 后加优化用
            nn.Linear(128, 10)
        )
    def forward(self, x):
      x = nn.Flatten()(x)
      output = self.nmodel(x)
      return output

device = ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"GPU->{device}")

model = NeuralNetwork().to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(x)
            print(f"loss: {loss:>7f} [{current:>5d}/{size:5d}]")



In [ ]:
def test(dataloader, model, loss_fn):
    num_batches = len(dataloader)
    size = len(dataloader.dataset)
    test_loss, correct = 0, 0
    model.eval()
    with torch.no_grad():
        for batch, (x, y) in enumerate(dataloader):
            x, y = x.to(device), y.to(device)
            pred = model(x)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        test_loss /= num_batches
        correct /= size

    print(f"test Error: \n Accuracy: {(100*correct):>0.1f}%, avg loss: {test_loss: > 8f}")



In [ ]:
import time

epochs = 20
start_time = time.time()

for epoch in range(epochs):
    print(f"{epoch+1}/{epochs}")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

elapsed = time.time() - start_time
print(f"Finished traning! Total time: {elapsed:.2f}s ({elapsed/60:.2f} min)")

In [ ]:
torch.save(model.state_dict(), "model_weights.pth")
print("saved pytorch model")